<a href="https://colab.research.google.com/github/tasinja/BankMarketingProject/blob/main/BankMarketingData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Βήμα 1: Αρχικοποίηση Περιβάλλοντος & Φόρτωση Δεδομένων
Σε αυτό το πρώτο στάδιο γίνεται η ρύθμιση του περιβάλλοντος Big Data. Συγκεκριμένα:
1. Εγκαθιστούμε τη βιβλιοθήκη `pyspark` στο Google Colab.
2. Συνδέουμε το Notebook με το Google Drive για να αντλήσουμε το διαμοιρασμένο αρχείο της ομάδας.
3. Ξεκινάμε ένα `SparkSession`, τον πυρήνα δηλαδή που θα αναλάβει την κατανεμημένη επεξεργασία των δεδομένων.
4. Φορτώνουμε το `bank-additional-full.csv` σε ένα Spark DataFrame, αναγνωρίζοντας αυτόματα τους τύπους των στηλών (inferSchema).



In [12]:
# 1. Εγκατάσταση του PySpark
!pip install pyspark

# 2. Σύνδεση με το Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 3. Εκκίνηση Spark Session
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("BankMarketingProject").getOrCreate()

# 4. Φόρτωση Δεδομένων
# (Αν το έβαλες χύμα στο Drive, η διαδρομή είναι αυτή. Αν το έβαλες σε φάκελο, πρόσθεσε το όνομα του φακέλου)
path = "/content/drive/MyDrive/BigDataProject/bank-additional-full.csv"

# Διαβάζουμε το αρχείο (έχει διαχωριστικό το ερωτηματικό ';')
df = spark.read.csv(path, header=True, inferSchema=True, sep=";")

# Τυπώνουμε τις 5 πρώτες γραμμές για να δούμε ότι δούλεψε!
df.show(5)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
+---+---------+-------+-----------+-------+-------+----+---------+-----+-----------+--------+--------+-----+--------+-----------+------------+--------------+-------------+---------+-----------+---+
|age|      job|marital|  education|default|housing|loan|  contact|month|day_of_week|duration|campaign|pdays|previous|   poutcome|emp.var.rate|cons.price.idx|cons.conf.idx|euribor3m|nr.employed|  y|
+---+---------+-------+-----------+-------+-------+----+---------+-----+-----------+--------+--------+-----+--------+-----------+------------+--------------+-------------+---------+-----------+---+
| 56|housemaid|married|   basic.4y|     no|     no|  no|telephone|  may|        mon|     261|       1|  999|       0|nonexistent|         1.1|        93.994|        -36.4|    4.857|     5191.0| no|
| 57| services|married|high.school|unknown|     no|  no|telephone|  may|       

### Βήμα 2: Προετοιμασία Δεδομένων (Data Preprocessing) - *Data Engineer*
Τα μοντέλα Μηχανικής Μάθησης δεν μπορούν να επεξεργαστούν κείμενο. Επομένως:
1. Χωρίζουμε τις μεταβλητές σε κατηγορικές (κείμενο) και αριθμητικές.
2. Μετατρέπουμε τον τελικό μας στόχο `y` (κατάθεση: yes/no) σε `1.0` και `0.0` (στήλη `label`).
3. Εφαρμόζουμε `StringIndexer` για να μετατρέψουμε κάθε λέξη (π.χ. "married", "student") σε αριθμητικό δείκτη.
4. Χρησιμοποιούμε τον `VectorAssembler` για να συγχωνεύσουμε όλα τα χαρακτηριστικά (features) κάθε πελάτη σε ένα ενιαίο διάνυσμα, το οποίο είναι η απαιτούμενη μορφή εισόδου για τους αλγόριθμους του Spark MLlib.

In [ ]:
from pyspark.ml.feature import StringIndexer, VectorAssembler

print("Ξεκινάει η προετοιμασία δεδομένων...")

# --- Η ΔΙΟΡΘΩΣΗ ΓΙΑ ΤΟ ΣΦΑΛΜΑ ---
# Αλλάζουμε όλες τις τελείες (.) στα ονόματα των στηλών σε κάτω παύλες (_)
for col_name in df.columns:
    df = df.withColumnRenamed(col_name, col_name.replace(".", "_"))

# 1. Λέμε στο πρόγραμμα ποιες στήλες έχουν κείμενο και ποιες νούμερα
categorical_columns = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']

# Προσοχή: Εδώ βάλαμε τα νέα ονόματα με τις κάτω παύλες!
numeric_columns = ['age', 'duration', 'campaign', 'pdays', 'previous', 'emp_var_rate', 'cons_price_idx', 'cons_conf_idx', 'euribor3m', 'nr_employed']

# 2. Μετατρέπουμε τον Στόχο μας (στήλη 'y') σε 0.0 (όχι) και 1.0 (ναι)
indexer_y = StringIndexer(inputCol="y", outputCol="label")
df_clean = indexer_y.fit(df).transform(df)

# 3. Μετατρέπουμε όλες τις υπόλοιπες λέξεις σε αριθμούς
for col in categorical_columns:
    indexer = StringIndexer(inputCol=col, outputCol=col+"_index")
    df_clean = indexer.fit(df_clean).transform(df_clean)

# 4. Πακετάρουμε τα πάντα σε μία τελική στήλη που λέγεται 'features'
assembler_inputs = numeric_columns + [c + "_index" for c in categorical_columns]
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features")

final_df = assembler.transform(df_clean)

print("Η προετοιμασία ολοκληρώθηκε! Έτσι βλέπει τα δεδομένα πλέον το Μοντέλο:")
final_df.select("features", "label").show(5, truncate=False)

Ξεκινάει η προετοιμασία δεδομένων...


### Βήμα 3: Εξερευνητική Ανάλυση & Γραφήματα (EDA) - *Data Analyst*
Εδώ εξάγουμε τα πρώτα οπτικά συμπεράσματα (Insights) από τα δεδομένα:
* Μετατρέπουμε το dataset σε `Pandas DataFrame` για συμβατότητα με τις βιβλιοθήκες οπτικοποίησης.
* Χρησιμοποιούμε `matplotlib` και `seaborn` για να δημιουργήσουμε:
  1. Ένα ραβδόγραμμα της κατανομής των κλάσεων (Ισορροπία του dataset: βλέπουμε ότι οι περισσότεροι λένε "Όχι").
  2. Ένα ιστόγραμμα που δείχνει τη σχέση μεταξύ της ηλικίας των πελατών και της πιθανότητας να κάνουν κατάθεση.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("Φόρτωση γραφημάτων...")

# Μετατρέπουμε τα αρχικά δεδομένα σε Pandas για να μπορούμε να τα ζωγραφίσουμε
pdf = df.toPandas()

# --- ΓΡΑΦΗΜΑ 1: Πόσοι είπαν Ναι και πόσοι Όχι; ---
plt.figure(figsize=(8,5))
sns.countplot(data=pdf, x='y', palette='Set2')
plt.title('Κατανομή Πελατών: Έκαναν κατάθεση (y) ;')
plt.ylabel('Αριθμός Πελατών')
plt.show()

# --- ΓΡΑΦΗΜΑ 2: Ηλικία σε σχέση με την Κατάθεση ---
plt.figure(figsize=(10,6))
sns.histplot(data=pdf, x='age', hue='y', multiple="stack", bins=30, palette='viridis')
plt.title('Ποιες ηλικίες τείνουν να κάνουν κατάθεση;')
plt.xlabel('Ηλικία')
plt.ylabel('Συχνότητα')
plt.show()

### Βήμα 4: Εκπαίδευση & Αξιολόγηση Μοντέλων - *ML Engineer*
Σε αυτό το τελικό στάδιο απαντάμε στο ερώτημα: *Μπορούμε να προβλέψουμε αν ένας πελάτης θα κάνει κατάθεση;*
1. **Διαχωρισμός Δεδομένων:** Χωρίζουμε τα δεδομένα μας σε σύνολο Εκπαίδευσης (Train - 80%) και σύνολο Ελέγχου (Test - 20%).
2. **Εκπαίδευση:** Χτίζουμε 2 διαφορετικά Classification μοντέλα (όπως ζητάει η εκφώνηση): Ένα Δέντρο Απόφασης (Decision Tree) και μια Λογιστική Παλινδρόμηση (Logistic Regression).
3. **Αξιολόγηση:** Επειδή τα δεδομένα μας είναι ανισόρροπα (imbalanced), το απλό Accuracy δεν είναι καλή μετρική. Χρησιμοποιούμε τη μετρική **AUC (Area Under ROC)**, όπου βαθμολογία κοντά στο 1.0 σημαίνει τέλειο μοντέλο.

In [ ]:
from pyspark.ml.classification import DecisionTreeClassifier, LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

print("Ξεκινάει η Εκπαίδευση των Μοντέλων... (Αυτό μπορεί να πάρει 1-2 λεπτά)")

# 1. Διαχωρισμός (80% Train, 20% Test)
train_data, test_data = final_df.randomSplit([0.8, 0.2], seed=42)

# 2. Μοντέλο 1: Decision Tree
dt = DecisionTreeClassifier(featuresCol="features", labelCol="label")
dt_model = dt.fit(train_data)
dt_predictions = dt_model.transform(test_data)

# 3. Μοντέλο 2: Logistic Regression
lr = LogisticRegression(featuresCol="features", labelCol="label", threshold=0.3)
lr_model = lr.fit(train_data)
lr_predictions = lr_model.transform(test_data)

# 4. Αξιολόγηση με AUC (Area Under ROC)
evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

dt_auc = evaluator.evaluate(dt_predictions)
lr_auc = evaluator.evaluate(lr_predictions)

print("\n--- ΤΕΛΙΚΑ ΑΠΟΤΕΛΕΣΜΑΤΑ (ΣΥΓΚΡΙΣΗ ΜΟΝΤΕΛΩΝ) ---")
print(f"Απόδοση Δέντρου Απόφασης (AUC): {dt_auc:.3f}")
print(f"Απόδοση Λογιστικής Παλινδρόμησης (AUC): {lr_auc:.3f}")

if lr_auc > dt_auc:
    print("Συμπέρασμα: Η Λογιστική Παλινδρόμηση αποδίδει καλύτερα!")
else:
    print("Συμπέρασμα: Το Δέντρο Απόφασης αποδίδει καλύτερα!")

### Βήμα 5: Αναλυτική Αξιολόγηση & Advanced Technique (K-Means)
Για να ολοκληρώσουμε την ανάλυσή μας, εμβαθύνουμε στο μοντέλο που κέρδισε (Λογιστική Παλινδρόμηση) και εφαρμόζουμε μια προχωρημένη τεχνική Μηχανικής Μάθησης:
1. **Αναλυτικά Μετρικά:** Εξάγουμε το Confusion Matrix (Πίνακας Σύγχυσης), το Accuracy, το Precision και το Recall για να δούμε ακριβώς πόσα λάθη έκανε το μοντέλο μας.
2. **Advanced Technique (Clustering):** Εφαρμόζουμε τον αλγόριθμο K-Means (Μη Επιβλεπόμενη Μάθηση) στα δεδομένα μας. Ζητάμε από τον αλγόριθμο να χωρίσει τους πελάτες σε 2 ομάδες (clusters) "στα τυφλά", για να δούμε αν μπορεί να αναγνωρίσει κρυφά μοτίβα χωρίς να ξέρει ποιος έκανε κατάθεση.

In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.clustering import KMeans

print("--- 1. Αναλυτική Αξιολόγηση του Αλγόριθμου 'Decision Tree ---")
# Υπολογισμός των στοιχείων του Πίνακα Σύγχυσης για το DT
tp_dt = dt_predictions.filter((dt_predictions.label == 1.0) & (dt_predictions.prediction == 1.0)).count()
tn_dt = dt_predictions.filter((dt_predictions.label == 0.0) & (dt_predictions.prediction == 0.0)).count()
fp_dt = dt_predictions.filter((dt_predictions.label == 0.0) & (dt_predictions.prediction == 1.0)).count()
fn_dt = dt_predictions.filter((dt_predictions.label == 1.0) & (dt_predictions.prediction == 0.0)).count()

print("\nΠίνακας Σύγχυσης (Decision Tree)")
dt_predictions.groupBy('label', 'prediction').count().show()

# Υπολογισμός Μετρικών
precision_dt = tp_dt / (tp_dt + fp_dt) if (tp_dt + fp_dt) != 0 else 0.0
recall_dt = tp_dt / (tp_dt + fn_dt) if (tp_dt + fn_dt) != 0 else 0.0
accuracy_dt = (tp_dt + tn_dt) / dt_predictions.count()

print(f"Accuracy (Ακρίβεια): {accuracy_dt:.3f}")
print(f"Precision (Θετική Προγνωστική Αξία): {precision_dt:.3f}")
print(f"Recall (Ευαισθησία): {recall_dt:.3f}")

print("--- 2. Αναλυτική Αξιολόγηση του Αλγόριθμου 'Λογιστική Παλινδρόμηση' ---")

# Confusion Matrix (Πίνακας Σύγχυσης)
print("\nΠίνακας Σύγχυσης (Confusion Matrix):")
# Στήλη label = Τι έγινε στην πραγματικότητα | Στήλη prediction = Τι προέβλεψε το μοντέλο
lr_predictions.crosstab('label', 'prediction').show()

# Υπολογισμός Accuracy, Precision & Recall
evaluator_multi = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")

accuracy = evaluator_multi.evaluate(lr_predictions, {evaluator_multi.metricName: "accuracy"})
# Υπολογίζουμε Precision και Recall ειδικά για την κλάση 1.0 (αυτούς που έκαναν κατάθεση)
precision = evaluator_multi.evaluate(lr_predictions, {evaluator_multi.metricName: "precisionByLabel", evaluator_multi.metricLabel: 1.0})
recall = evaluator_multi.evaluate(lr_predictions, {evaluator_multi.metricName: "recallByLabel", evaluator_multi.metricLabel: 1.0})

print(f"Accuracy (Ακρίβεια): {accuracy:.3f}")
print(f"Precision (Θετική Προγνωστική Αξία): {precision:.3f}")
print(f"Recall (Ευαισθησία): {recall:.3f}")


print("\n--- 2. ADVANCED TECHNIQUE: CLUSTERING (K-MEANS) ---")
# Εκπαίδευση του K-Means αλγορίθμου για να χωρίσει τους πελάτες σε 2 ομάδες (clusters)
kmeans = KMeans(featuresCol="features", predictionCol="cluster", k=2, seed=42)
kmeans_model = kmeans.fit(final_df)
clusters_df = kmeans_model.transform(final_df)

print("\nΟ αλγόριθμος K-Means χώρισε τους πελάτες στις εξής 2 ομάδες (Clusters):")
clusters_df.groupBy("cluster").count().show()
print("Συμπέρασμα: Ο K-Means βρήκε μια τεράστια ομάδα πελατών με παρόμοια χαρακτηριστικά και μια μικρότερη, επιβεβαιώνοντας την ανισορροπία των δεδομένων μας!")

### Βήμα 6: Τελική Σύγκριση Μοντέλων (AUC - ROC Curve)

Για να ολοκληρώσουμε την εργασία και να δικαιολογήσουμε επιστημονικά την τελική μας επιλογή, συγκρίνουμε τα δύο classification μοντέλα (Decision Tree vs. Logistic Regression) χρησιμοποιώντας τη μετρική AUC (Area Under ROC):

In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Δημιουργία του αξιολογητή για το εμβαδόν (AUC)
evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

# 1. Υπολογισμός AUC για το Δέντρο Απόφασης
auc_dt = evaluator.evaluate(dt_predictions)
print(f"Το Εμβαδόν AUC για το Decision Tree είναι: {auc_dt:.3f}")

# 2. Υπολογισμός AUC για τη Λογιστική Παλινδρόμηση
auc_lr = evaluator.evaluate(lr_predictions)
print(f"Το Εμβαδόν AUC για τη Logistic Regression είναι: {auc_lr:.3f}")